# Morphological Feature Extraction for Tumor Architecture Masks

**Author: Fabien Bertillot**

This notebook computes the mask-level architecture features of the paper from tumour–stroma segmentation masks, such as those written by `tumor-stroma_mask_prediction.ipynb`. It returns one row per TMA core. The feature-extraction functions are in `morphology_utils.py`.

The tumour mask is cleaned by removing objects smaller than 20 px and filling holes smaller than 100 px. Then:

- **Branch length, branch types and straightness:** the mask is skeletonized (`skimage.morphology.skeletonize`), branches shorter than 5 px are pruned with `skan`, and branch arc lengths (`branch-distance`), skan branch types (proportions of types 0, 1 and 2) and straightness (end-to-end distance / arc length) are summarized per core.
- **Skeleton cycles:** number and length of the cycles of the pruned skeleton graph.
- **Lumens:** holes enclosed by the tumour mask (`scipy.ndimage.binary_fill_holes`), counted and measured with `skimage.measure.regionprops`.
- **Contour roughness:** mean absolute curvature per contour point of the Gaussian-smoothed (σ = 1) tumour contours.
- **Tumour–stroma ratio (`tumor_area`):** tumour pixel count / stroma pixel count, and `lumen_ratio` = lumen count / `tumor_area`.

Cell-neighbourhood features, including the fragmentation index, are computed in `neighborhood_feature_extraction.ipynb`.

---

## 1) Load Dependencies

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage.morphology import skeletonize

from morphology_utils import (
    batch_measure_architecture_morphology,
    preprocess_tumor_mask,
    prune_skeleton,
)

## 2) Configure Inputs

Set `MASK_DIR` to the folder of tumour–stroma masks and give the label values of tumour and stroma. The masks written by `tumor-stroma_mask_prediction.ipynb` use `1` for stroma and `2` for tumour; the masks behind the paper tables used `1` for tumour and `2` for stroma.

The size thresholds are in pixels and match the paper.

In [ ]:
MASK_DIR = Path(r"../outputs/architecture/tumor_stroma_masks")
OUT_DIR = Path(r"../outputs/architecture/morphology")

MASK_GLOB = "*.png"
MASK_SUFFIX = ""  # Removed from the file name to give image_name, for example "_Simple Segmentation"
TUMOR_LABEL = 2
STROMA_LABEL = 1

MIN_OBJECT_AREA = 20
MIN_HOLE_AREA = 100
MIN_BRANCH_LENGTH = 5

In [ ]:
# papermill passes parameters as strings
MASK_DIR = Path(MASK_DIR)
OUT_DIR = Path(OUT_DIR)

OUT_DIR.mkdir(parents=True, exist_ok=True)
mask_paths = sorted(MASK_DIR.glob(MASK_GLOB))
print(f"Found {len(mask_paths)} masks")

if not mask_paths:
    raise FileNotFoundError(f"No masks found in {MASK_DIR} with pattern {MASK_GLOB!r}")

## 3) Preview One Mask

This quick visual check catches swapped tumour/stroma labels or unexpected file naming before running the batch extraction.

In [ ]:
example_path = mask_paths[0]
example_mask = np.array(Image.open(example_path))
example_tumor = preprocess_tumor_mask(example_mask == TUMOR_LABEL, MIN_OBJECT_AREA, MIN_HOLE_AREA)
example_skeleton, _, _ = prune_skeleton(skeletonize(example_tumor), MIN_BRANCH_LENGTH)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(example_mask == TUMOR_LABEL, cmap="gray")
axes[0].set_title("Tumour")
axes[1].imshow(example_mask == STROMA_LABEL, cmap="gray")
axes[1].set_title("Stroma")
axes[2].imshow(example_tumor, cmap="gray")
axes[2].imshow(np.ma.masked_where(~example_skeleton, example_skeleton), cmap="autumn")
axes[2].set_title("Pruned skeleton")

for ax in axes:
    ax.axis("off")

plt.suptitle(example_path.name)
plt.tight_layout()

## 4) Extract Morphology Features

Cores without lumens, or with too few lumens or skeleton cycles for a skewness, get `NaN` for some summary statistics. As in the paper tables, these are set to 0.

In [ ]:
df_morphology = batch_measure_architecture_morphology(
    mask_paths,
    tumor_label=TUMOR_LABEL,
    stroma_label=STROMA_LABEL,
    mask_suffix=MASK_SUFFIX,
    min_object_area=MIN_OBJECT_AREA,
    min_hole_area=MIN_HOLE_AREA,
    min_branch_length=MIN_BRANCH_LENGTH,
)

nan_to_zero_columns = ["cycle_length_skew", "lumen_area_mean", "lumen_area_skew", "lumen_area_coef_variation"]
df_morphology[nan_to_zero_columns] = df_morphology[nan_to_zero_columns].fillna(0)

df_morphology.head()

## 5) Feature Sanity Checks

These plots provide a fast check for outliers and empty masks before exporting the table.

In [ ]:
plot_columns = [
    "tumor_area",
    "branch_length_mean",
    "prop_of_type2_branches",
    "lumen_count",
    "lumen_area_mean",
    "roughness",
]

axes = df_morphology[plot_columns].hist(figsize=(14, 8), bins=30)
plt.tight_layout()

## 6) Save Feature Table

In [ ]:
out_csv = OUT_DIR / "architecture_morphology_features.csv"
df_morphology.to_csv(out_csv, index=False)
print(f"Saved {len(df_morphology)} rows to {out_csv}")